# v65 生PF提出 — **struct20 + dec10 の 50:50 平均**
- set0 = struct20(struct錨 mult20, grid0.2, np600)を **MD-1**(そのまま)で PF。単体 LB≈7.9。
- set1 = dec10(mult12, grid0.5, np400, 探索best 7.566)を **MD-10 集約→全解像度に補間**して PF。`pf_dec_v65` と同一ロジック。
- 各wellで 2軌跡を **nanmean(=50:50)**。CV誤差相関 0.802 の直交ペア(CV blend ~7.11, 単体struct20比 −0.33)。
- struct は v66_leanv1cumul(実行時構築)。dataset に `struct_field_v66_leanv1cumul.npz` を添付。


In [ ]:
# ===== 設定(ここだけ編集) =====
import os, json, time, glob as _glob, math, warnings, pickle
import numpy as np, pandas as pd
from pathlib import Path
import torch
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

# --- PFパラメータセット(ハードコード。tune_pf_v65 の params をそのまま貼る) ---
PARAM_SETS = [
    # set0: struct20 (mult20, grid0.2, np600) — MD-1
    {"use_phys": False, "use_anchor": True, "anchor_ramp_md": 0.0, "lk_floor": 1e-05, "gr_debias": 0, "rate_target": "zero", "grid_step": 0.2, "tw_source": "tw", "self_mix_w": 1.0, "nbr_mix_w": 1.0, "n_particles": 600, "likelihood_scale": 29.056396339290437, "init_pos_std": 1.6168048868046325, "init_rate_std": 0.014412715870661067, "mom": 0.9995682504305502, "vn": 0.000873883338794689, "pn": 0.00606209538025288, "rp": 0.14025726060525087, "rr": 0.00398567931602027, "resamp": 0.18645022443117387, "gr_sig_min": 10.63834771185878, "gr_sig_max": 43.95858017409127, "gr_sig_def": 86.87561008087175, "gr_sig_mult": 2.7681905590609426, "tvt_clip_margin": 86.30698497112195, "rate_clip": 0.0, "gr_power": 1.85175446220882, "hgr_smooth_r": 2, "tw_gr_smooth_r": 0, "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 2.3072090423765363, "anchor_mult": 20.0},
    # set1: dec10 (mult12.2, grid0.5, np400, tune7.566) — MD-10間引き
    {"use_phys": False, "use_anchor": True, "anchor_ramp_md": 0.0, "lk_floor": 0.0, "gr_debias": 0, "rate_target": "zero", "grid_step": 0.5, "tw_source": "tw", "self_mix_w": 1.0, "nbr_mix_w": 1.0, "anchor_mult": 12.162649267458692, "n_particles": 400, "likelihood_scale": 9.096036883598687, "init_pos_std": 1.6244748366473936, "init_rate_std": 0.001524769837129659, "mom": 0.997562411801969, "vn": 0.0021783331559066133, "pn": 0.002754175767599275, "rp": 0.10011641574625953, "rr": 0.00038802129847735704, "resamp": 0.3302631233337396, "gr_sig_min": 7.803949460564919, "gr_sig_max": 22.073371866038766, "gr_sig_def": 132.7732967342783, "gr_sig_mult": 1.6432611268042199, "tvt_clip_margin": 135.04931448640443, "rate_clip": 0.0, "gr_power": 1.980155629637672, "hgr_smooth_r": 2, "tw_gr_smooth_r": 1, "jump_prob": 0.02, "jump_std": 5.0, "phys_sig": 0.038303481512635865},
]
DEC_SETS = [1, 10]   # 各セットの間引き倍率(1=MD-1そのまま / 10=MD-10集約→全解像度に補間)
N_SEEDS = 32          # seedアンサンブル(尤度加重平均)
SEED    = 777
AVG     = "mean"      # 複数セットの合成: "mean" | "median"
PP      = None        # 例 dict(tau=40.0, alpha=1.03): d=(pf-lk)*(1-exp(-mds/tau))*alpha。None=生値
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.float32
CHUNK   = 60

def _detect_data():
    for _p in ["/kaggle/input/rogii-wellbore-geology-prediction",
               "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
               r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction"]:
        if os.path.exists(_p): return Path(_p)
    _h = _glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    return Path(os.path.dirname(_h[0]))
DATA = _detect_data(); TESTD = DATA / "test"
print("DATA:", DATA, "| device:", DEVICE, "| sets:", len(PARAM_SETS))

In [ ]:
# ===== PFエンジン(pf_engine_v65.py から抽出した純計算部) =====
def _bin_pairs(tvt, gr, gmin, step, G):
    """(TVT,GR)点群 → 格子bin中央値(grid_step共通)。無データbin=NaN。"""
    out = np.full(G, np.nan)
    idx = np.round((np.asarray(tvt, float) - gmin) / step).astype(int)
    ok = (idx >= 0) & (idx < G) & np.isfinite(gr) & np.isfinite(tvt)
    if ok.sum():
        srt = pd.Series(np.asarray(gr, float)[ok]).groupby(idx[ok]).median()
        out[srt.index.to_numpy()] = srt.to_numpy()
    return out

def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2 * r + 1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b + step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf, step=float(P.get("grid_step", 0.2)))
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    bias = 0.0
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
        if int(P.get("gr_debias", 0)):
            b_ = float(np.nanmedian(resid))
            if np.isfinite(b_): bias = b_       # 井戸レベルGRオフセット(工具較正差)を尤度前に補正
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    if bias != 0.0:
        gr_sm = gr_sm - bias
    # ---- self/neighbor 疑似typewell混合(σ較正後。grid_step は全格子共通) ----
    src = str(P.get("tw_source", "tw"))
    if src != "tw":
        gg = gg.copy()
        if "nbr" in src and c.get("nbr_pairs") is not None:
            nt, ng = c["nbr_pairs"]
            ngs = _smooth(ng, float(np.nanmean(ng)), P["hgr_smooth_r"]).astype(np.float64)
            gh = _bin_pairs(nt, ngs, gmin, gst, len(gg))
            have = np.isfinite(gh); w = float(P.get("nbr_mix_w", 1.0))
            gg[have] = w * gh[have] + (1 - w) * gg[have]
        if "self" in src:
            gh = _bin_pairs(c["kn_tvtin"], gr_sm[c["kpos"]].astype(np.float64), gmin, gst, len(gg))
            have = np.isfinite(gh); w = float(P.get("self_mix_w", 1.0))
            gg[have] = w * gh[have] + (1 - w) * gg[have]
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]),
                anc=c["anc"], ancs=c["ancs"])

def _pf_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, anc, ancs, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    USE_A = bool(P.get("use_anchor", True)); AMUL = float(P.get("anchor_mult", 20.0))
    RAMP = float(P.get("anchor_ramp_md", 0.0))          # >0: 錨をmdsでランプイン(近錨域の引きずり防止)
    LKF = float(P.get("lk_floor", 0.0))                  # >0: 尤度の一様混合床(外れGRロバスト化)
    RT_IR = (str(P.get("rate_target", "zero")) == "ir")  # momentum減衰先: 0 or 初期dip
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + ((1.0 - ALPHA) * dipp if RT_IR else 0.0) + RN * rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if LKF > 0:
            lk = (1.0 - LKF) * lk + LKF                  # 一様混合床: 外れGRで正解粒子が即死しない
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        if USE_A:                                   # ★ row錨(毎行独立掛け, mult=温度補正, ramp=遅延起動)
            asig = (ancs[:, i][:, None] * AMUL).clamp_min(1e-6)
            da = (v - anc[:, i][:, None]) / asig    # v = pos - zi = 粒子のTVT
            da2 = da * da
            if RAMP > 0:
                tf = ((cur - md[:, 0]).clamp_min(0.0) / RAMP).clamp(0.0, 1.0)[:, None]
                da2 = da2 * tf                       # 温度でランプイン(近錨域は錨を弱く)
            lka = torch.where(da2 < 1200.0, torch.exp(-0.5 * da2), zero).clamp_min(1e-300)
        else:
            lka = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
        pts[:, i] = (w * (pos - zi)).sum(1); pm = torch.where(act, cur, pm)
    return pts, log_lik

def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
        anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
        if Tn < Tmax: anc[b, Tn:] = anc[b, Tn - 1]
        ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs))

def run_forward_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pts, ll = _pf_core(rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                       pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                       pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                       rep(pad["anc"]), rep(pad["ancs"]), P, N, device, gen)
    pts = pts.view(W, S, -1).double().cpu().numpy(); ll = ll.view(W, S).cpu().numpy()
    outs = []
    for w in range(W):
        T = len(inps[w]["md"]); pr = pts[w, :, :T]
        llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6))
            s = ww.sum()
            ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
        else:
            ww = np.full(S, 1.0 / S)
        outs.append((ww[:, None] * pr).sum(0))
    return outs


# ----------------------- 評価(対象well指定) -----------------------

In [ ]:
# ===== test側キャッシュ + 構造prior(v66_leanv1cumul: 回帰場 VEC + PGATE + 累積σ) =====
NEED_ANCHOR = any(bool(P.get("use_anchor", False)) for P in PARAM_SETS)
STRUCT_T = {}
if NEED_ANCHOR:
    _fz = next((p for p in (_glob.glob("/kaggle/input/**/struct_field_v66_leanv1cumul.npz", recursive=True)
                            + [r"C:\Users\kosaka256\Documents\rogii_claude\v66\artifacts_v66\struct_field_v66_leanv1cumul.npz"]) if os.path.exists(p)), None)
    if _fz is None:
        print("★警告: struct_field_v66_leanv1cumul.npz が見つからない → 錨セットは錨OFFで実行")
    else:
        FZ = np.load(_fz, allow_pickle=True)
        FX = FZ["x"].astype(np.float64); FY = FZ["y"].astype(np.float64)
        FC = FZ["c"].astype(np.float64); FAZ = FZ["az"].astype(np.float64)
        ALPHA_ = np.asarray(FZ["alpha"], np.float64); BETA_ = np.asarray(FZ["beta"], np.float64)
        AP_ = float(FZ["ap"]); BP_ = float(FZ["bp"])
        BASEV = float(FZ["base_var"]); RHO = np.asarray(FZ["rho"], np.float64); RHO_SP = float(FZ["rho_spread"])
        KB = np.asarray(FZ["kbins"], np.float64)
        _cfg = json.loads(str(FZ["config"]))
        TH = float(_cfg["theta0"]); KK = int(_cfg["knn_k"]); KH = float(_cfg["knn_h"])
        CMAX = float(_cfg["cmax"]); PGATE = float(_cfg["pgate"]); VEC = int(_cfg["vec"]); SEG_LEN = float(_cfg["seg_len"])
        SMIN = float(_cfg["sig_min"]); SCAP = float(_cfg.get("sig_cap", 40.0))
        NBIN_ = len(KB) - 1; K_FIX = 16; K_SEG_MAX = 64
        C0_ = np.cos(np.radians(TH)); S0_ = np.sin(np.radians(TH))
        FP0 = FX * 0 + np.cos(FAZ - np.radians(TH))   # ドナー方位のθ0射影 cos(az-θ0)
        FUX = np.cos(FAZ); FUY = np.sin(FAZ)

        def _seg_test(Xl, Yl, arc, n):
            total = float(arc[-1])
            if SEG_LEN <= 0:
                Kn = min(K_FIX, max(1, n // 4))
            else:
                Kn = int(np.clip(round(total / SEG_LEN), 1, min(K_SEG_MAX, max(1, n // 24))))
            edges = np.linspace(0.0, total, Kn + 1)
            segid = np.clip(np.searchsorted(edges[1:], arc, side="left"), 0, Kn - 1)
            mid = np.empty((Kn, 2)); az = np.empty(Kn)
            for j in range(Kn):
                rows = np.where(segid == j)[0]
                j0, j1 = (int(rows[0]), int(rows[-1])) if len(rows) else (0, n - 1)
                az[j] = np.arctan2(Yl[j1 + 1] - Yl[j0], Xl[j1 + 1] - Xl[j0])
                mid[j] = ((Xl[j0] + Xl[j1 + 1]) / 2.0, (Yl[j0] + Yl[j1 + 1]) / 2.0)
            return segid, mid, az

        def _query_test(mids):
            Kn = len(mids); g = np.zeros((Kn, 2)); dd = np.empty(Kn); spread = np.zeros(Kn)
            for j in range(Kn):
                d2 = (FX - mids[j, 0]) ** 2 + (FY - mids[j, 1]) ** 2
                kk = min(KK, len(d2) - 1) if len(d2) > 1 else 1
                sel = np.argpartition(d2, kk - 1)[:kk]
                w = np.exp(np.maximum(-d2[sel] / (2 * KH ** 2), -700.0))   # qw=1(QW off)
                ws = float(w.sum()); dd[j] = np.sqrt(np.median(d2[sel]))
                if ws <= 1e-300:
                    continue
                p0 = FP0[sel]; cc = FC[sel]
                s_hat = float((w * cc * p0).sum() / ((w * p0 * p0).sum() + 1e-9))
                gx, gy = s_hat * C0_, s_hat * S0_
                if VEC >= 2:                             # 2Dベクトル場(leanv1cumulはVEC=1で不使用)
                    ux = FUX[sel]; uy = FUY[sel]
                    Sxx = (w * ux * ux).sum(); Sxy = (w * ux * uy).sum(); Syy = (w * uy * uy).sum()
                    bx = (w * cc * ux).sum(); by = (w * cc * uy).sum()
                    tr = Sxx + Syy; det = Sxx * Syy - Sxy * Sxy; disc = np.sqrt(max(tr * tr - 4 * det, 0.0))
                    lam1 = (tr + disc) / 2; lam2 = (tr - disc) / 2; ratio = lam2 / max(lam1, 1e-12)
                    lam = lam1 * max(0.05, 0.25 - ratio)
                    A11 = Sxx + lam; A22 = Syy + lam; A12 = Sxy; b1 = bx + lam * gx; b2 = by + lam * gy
                    detA = A11 * A22 - A12 * A12
                    gx = (b1 * A22 - b2 * A12) / max(detA, 1e-12); gy = (b2 * A11 - b1 * A12) / max(detA, 1e-12)
                gn = math.hypot(gx, gy)
                if gn > CMAX: gx *= CMAX / gn; gy *= CMAX / gn
                g[j, 0] = gx; g[j, 1] = gy
                if RHO_SP:                                # ドナー不一致(spread変種のみ)
                    resd = cc - (gx * FUX[sel] + gy * FUY[sel])
                    spread[j] = np.sqrt(max(float((w * resd ** 2).sum() / max(ws, 1e-12)), 0.0))
            return g, dd, spread

        for fp in _glob.glob(str(TESTD / "*__horizontal_well.csv")):
            wid = os.path.basename(fp).split("__")[0]
            try:
                df = pd.read_csv(fp, usecols=["X", "Y", "Z", "TVT_input"])
                X = df["X"].to_numpy(float); Y = df["Y"].to_numpy(float); Z = df["Z"].to_numpy(float); ti = df["TVT_input"].to_numpy(float)
                fin = np.where(np.isfinite(ti))[0]
                if len(fin) == 0: continue
                s = int(fin.max()); n = len(X) - 1 - s
                if n < 2: continue
                dX = np.diff(X)[s:]; dY = np.diff(Y)[s:]; ndz = -np.diff(Z)[s:]
                Lxy = np.maximum(np.hypot(dX, dY), 1e-3); arc = np.cumsum(Lxy)
                Xl = X[s:]; Yl = Y[s:]
                p_row = (dX * C0_ + dY * S0_) / Lxy
                perp = (np.abs(p_row) < PGATE) if PGATE > 0 else np.zeros(n, bool)
                segid, mid, az = _seg_test(Xl, Yl, arc, n)
                g, dd, spread = _query_test(mid)
                f = g[segid, 0] * dX + g[segid, 1] * dY
                b_row = np.digitize(dd, KB[1:-1])[segid]
                av = ALPHA_[b_row].copy(); bv = BETA_[b_row].copy()
                if PGATE > 0:
                    av = np.where(perp, AP_, av); bv = np.where(perp, BP_, bv)
                resid = np.cumsum(av * ndz + bv * f)
                tvt = float(ti[s]) + resid
                # 累積σ: var = base + Σ_{行≤i} rho(bin_j)[+ rho_spread·Σspread]
                dd_row = dd[segid]
                bb = np.digitize(dd_row, KB[1:-1]).astype(int)
                if PGATE > 0: bb = np.where(perp, NBIN_, bb)
                var = BASEV + np.cumsum(RHO[bb])
                if RHO_SP: var = var + RHO_SP * np.cumsum(spread[segid])
                sig = np.clip(np.sqrt(var), SMIN, SCAP)
                STRUCT_T[wid] = dict(tvt=tvt.astype(np.float64), sig=sig.astype(np.float64))
            except Exception:
                continue
        print(f"[struct v66_leanv1cumul] test実行時構築: {len(STRUCT_T)} wells")

# --- self/nbr 用: train近傍プール(nbrセットがある場合のみ構築。test近傍=train坑井, ≤1500ft, ≤3本, 2本未満=tw) ---
MAX_DIST = 1500.0; K_MAX = 3; NEED_REFS = 2
NEED_NBR = any("nbr" in str(P.get("tw_source", "tw")) for P in PARAM_SETS)
TRAIND = DATA / "train"
_TRC = None; _REFC = {}
def _train_centroids():
    global _TRC
    if _TRC is None:
        wids = sorted(set(os.path.basename(f).split("__")[0] for f in _glob.glob(str(TRAIND / "*__horizontal_well.csv"))))
        xy = []
        for w_ in wids:
            h_ = pd.read_csv(TRAIND / f"{w_}__horizontal_well.csv", usecols=["X", "Y"])
            xy.append((float(h_["X"].mean()), float(h_["Y"].mean())))
        _TRC = (wids, np.array(xy))
        print(f"[nbr] trainプール: {len(wids)} wells")
    return _TRC
def _ref_pairs(w_):
    if w_ not in _REFC:
        h_ = pd.read_csv(TRAIND / f"{w_}__horizontal_well.csv", usecols=["GR", "TVT_input"])
        m_ = h_["TVT_input"].notna().to_numpy() & np.isfinite(h_["GR"].to_numpy(float))
        _REFC[w_] = (h_["TVT_input"].to_numpy(float)[m_], h_["GR"].to_numpy(float)[m_])
    return _REFC[w_]
def _nbr_pairs_of(cx, cy):
    wids, xy = _train_centroids()
    d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy)
    order = np.argsort(d)
    refs = [wids[j] for j in order if d[j] <= MAX_DIST][:K_MAX]
    if len(refs) < NEED_REFS: return None
    ts = []; gs_ = []
    for r_ in refs:
        t_, g_ = _ref_pairs(r_); ts.append(t_); gs_.append(g_)
    return (np.concatenate(ts), np.concatenate(gs_))

def build_cache_test(wid):
    """v65 build_cache の test版(TVT不要)。返り値: make_inputs 互換 cache dict + 行index。"""
    hw = pd.read_csv(TESTD / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TESTD / f"{wid}__typewell.csv").sort_values("TVT")
    kn = hw["TVT_input"].notna(); ev = ~kn
    if kn.sum() < 10 or ev.sum() == 0: return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    fin = np.isfinite(tw_tvt) & np.isfinite(tw_gr); tw_tvt, tw_gr = tw_tvt[fin], tw_gr[fin]
    if len(tw_tvt) < 3: return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(hw[ev].index); kpos = hw.index.get_indexer(hw[kn].index)
    kn_tvtin = hw["TVT_input"].to_numpy(float)[kpos]
    ls = float(kn_tvtin[-1] + hw["Z"].to_numpy(float)[kpos][-1])
    tl = hw[kn].tail(30)
    dt = np.diff(tl["TVT_input"].values); dz = np.diff(tl["Z"].values); dm = np.diff(tl["MD"].values)
    m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    nev = int(ev.sum())
    sp = STRUCT_T.get(wid)
    if sp is not None and len(sp["tvt"]) == nev:
        anc = np.asarray(sp["tvt"], np.float64); ancs = np.asarray(sp["sig"], np.float64)
    else:
        anc = np.zeros(nev); ancs = np.full(nev, 1e9)
    nbr_pairs = _nbr_pairs_of(float(hw["X"].mean()), float(hw["Y"].mean())) if NEED_NBR else None
    return dict(md=hw["MD"].to_numpy(float)[ev_pos], z=hw["Z"].to_numpy(float)[ev_pos],
                gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos, kn_tvtin=kn_tvtin,
                tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb, ls=ls, ir=ir,
                anc=anc, ancs=ancs, ev_index=np.where(ev.to_numpy())[0], lk=float(kn_tvtin[-1]),
                nbr_pairs=nbr_pairs)

In [ ]:
# ===== 実行: struct20(MD-1) + dec10(MD-10間引き) を各々回して 50:50 平均 =====
# --- dec 集約ヘルパ(pf_dec_v65 と同一。単相 PHASE=1, GR=mean) ---
def _agg_dec(x, s, how="mean"):
    x = np.asarray(x, float); n = len(x)
    cuts = sorted(set([0, n] + list(range(s, n, s))))
    out = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        seg = x[a:b]; out.append(float(seg.mean()) if how == "mean" else float(seg[(b - a) // 2]))
    return np.array(out)
def make_inputs_dec_sub(cache, P, DEC):
    inp = make_inputs(cache, P); md = np.asarray(inp["md"], float)
    if len(md) < DEC * 2:
        di = dict(inp); di["full_md"] = md; di["dec_md"] = md; return di
    di = dict(md=_agg_dec(md, DEC), z=_agg_dec(inp["z"], DEC), gr=_agg_dec(inp["gr"], DEC),
              gg=inp["gg"], gmin=inp["gmin"], gst=inp["gst"], gs=inp["gs"], ls=inp["ls"], ir=inp["ir"],
              N=inp["N"], anc=_agg_dec(inp["anc"], DEC), ancs=_agg_dec(inp["ancs"], DEC))
    di["full_md"] = md; di["dec_md"] = di["md"]; return di

t0 = time.time()
wids = sorted(set(os.path.basename(f).split("__")[0] for f in _glob.glob(str(TESTD / "*__horizontal_well.csv"))))
caches = {}
for w in wids:
    c = build_cache_test(w)
    if c is not None: caches[w] = c
W = sorted(caches.keys())
print(f"test wells: {len(W)}/{len(wids)}")

preds_by_set = []
for si, P0 in enumerate(PARAM_SETS):
    P = dict(P0); DECi = int(DEC_SETS[si])
    outs = {}
    for c0_ in range(0, len(W), CHUNK):
        sub = W[c0_:c0_ + CHUNK]
        if DECi > 1:                                            # ★MD-DEC 集約→補間
            dinps = [make_inputs_dec_sub(caches[w], P, DECi) for w in sub]
            res = run_forward_ens(dinps, P, DEVICE, SEED, N_SEEDS)
            for j, w in enumerate(sub):
                dm = dinps[j]["dec_md"]; fm = dinps[j]["full_md"]; dt = np.asarray(res[j], np.float64)
                outs[w] = dt if len(dm) == len(fm) else np.interp(fm, dm, dt)   # 全解像度に補間
        else:                                                   # MD-1 そのまま
            inps = [make_inputs(caches[w], P) for w in sub]
            res = run_forward_ens(inps, P, DEVICE, SEED, N_SEEDS)
            for j, w in enumerate(sub): outs[w] = np.asarray(res[j], np.float64)
        if DEVICE == "cuda": torch.cuda.empty_cache()
    preds_by_set.append(outs)
    print(f"set{si+1}/{len(PARAM_SETS)} (dec{DECi}) done ({time.time()-t0:.0f}s)", flush=True)

pids = []; pvals = []
for w in W:
    c = caches[w]
    A = np.stack([ps[w] for ps in preds_by_set], 0)            # (2, nev) struct20 / dec10 の TVT軌跡
    z = np.nanmedian(A, 0) if AVG == "median" else np.nanmean(A, 0)   # ★50:50 平均
    if PP:
        mds = np.maximum(c["md"] - c["md"][0] + 1.0, 0.0)
        d = (z - c["lk"]) * (1.0 - np.exp(-mds / float(PP["tau"]))) * float(PP.get("alpha", 1.0))
        z = c["lk"] + d
    pids += [f"{w}_{int(i)}" for i in c["ev_index"]]
    pvals += list(z)
pred_df = pd.DataFrame({"id": pids, "tvt": pvals})
print("predicted rows:", len(pred_df), f"({time.time()-t0:.0f}s)")


In [ ]:
# ===== submission(欠け行は lk フォールバック) =====
ss = pd.read_csv(DATA / "sample_submission.csv")
sub = ss[["id"]].merge(pred_df, on="id", how="left")
miss = sub["tvt"].isna()
if miss.any():
    lkmap = {}
    for wid in set(i.rsplit("_", 1)[0] for i in sub.loc[miss, "id"]):
        h = pd.read_csv(TESTD / f"{wid}__horizontal_well.csv", usecols=["TVT_input"])
        v = h["TVT_input"].dropna()
        lkmap[wid] = float(v.iloc[-1]) if len(v) else 0.0
    sub.loc[miss, "tvt"] = [lkmap[i.rsplit("_", 1)[0]] for i in sub.loc[miss, "id"]]
    print(f"lkフォールバック: {int(miss.sum())} rows")
sub.to_csv("submission.csv", index=False)
print("submission.csv:", len(sub), "rows")